<a href="https://colab.research.google.com/github/m28851119-cmd/Deep-Clustering/blob/main/%E6%B7%B1%E5%BA%A6%E8%81%9A%E7%B1%BB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 导包


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import normalized_mutual_info_score
import numpy as np
from scipy.optimize import linear_sum_assignment

#  1. 环境与数据准备

In [4]:
# ==========================================
# 1. 环境与数据准备
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前运行设备: {device}")

# 图像预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# 使用 CIFAR-10 测试集（10000张）进行快速验证
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
dataloader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# 准备评估函数（匈牙利算法匹配标签）
def cluster_eval(y_true, y_pred):
    y_true = np.array(y_true).astype(np.int64)
    y_pred = np.array(y_pred).astype(np.int64)
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    acc = sum([w[i, j] for i, j in zip(row_ind, col_ind)]) / y_pred.size
    nmi = normalized_mutual_info_score(y_true, y_pred)
    return acc, nmi

# 提取原始数据用于传统算法
X_raw_list, Y_true_list = [], []
for imgs, labels in dataloader:
    X_raw_list.append(imgs.view(imgs.size(0), -1).numpy()) # 展平用于传统算法
    Y_true_list.append(labels.numpy())
X_raw = np.concatenate(X_raw_list)
Y_true = np.concatenate(Y_true_list)


当前运行设备: cuda


# 2. 方案一 & 方案二（传统方法）

In [5]:
# ==========================================
# 2. 方案一 & 方案二（传统方法）
# ==========================================
print("\n[方案一] 运行 K-means (原始像素)...")
km = KMeans(n_clusters=10, n_init=10, random_state=42)
acc1, nmi1 = cluster_eval(Y_true, km.fit_predict(X_raw))
print(f"结果: ACC={acc1:.4f}, NMI={nmi1:.4f}")

print("\n[方案二] 运行 PCA + K-means (50维)...")
X_pca = PCA(n_components=50).fit_transform(X_raw)
acc2, nmi2 = cluster_eval(Y_true, km.fit_predict(X_pca))
print(f"结果: ACC={acc2:.4f}, NMI={nmi2:.4f}")



[方案一] 运行 K-means (原始像素)...
结果: ACC=0.2036, NMI=0.0841

[方案二] 运行 PCA + K-means (50维)...
结果: ACC=0.2050, NMI=0.0847


# 3. 方案三：完整卷积 DEC

In [6]:
# ==========================================
# 3. 方案三：完整卷积 DEC
# ==========================================

# 3.1 定义卷积自编码器
class ConvAE(nn.Module):
    def __init__(self):
        super(ConvAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(), # 16x16
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(), # 8x8
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(), # 4x4
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128) # 嵌入层
        )
        self.decoder = nn.Sequential(
            nn.Linear(128, 128 * 4 * 4),
            nn.Unflatten(1, (128, 4, 4)),
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1), nn.Tanh()
        )
    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

# 3.2 定义 DEC 聚类层 (KL优化核心)
class ClusteringLayer(nn.Module):
    def __init__(self, n_clusters=10, latent_dim=128):
        super(ClusteringLayer, self).__init__()
        self.centers = nn.Parameter(torch.zeros(n_clusters, latent_dim))
    def forward(self, z):
        # 计算 Q 分布 (Student-t)
        dist = torch.sum((z.unsqueeze(1) - self.centers)**2, dim=2)
        q = 1.0 / (1.0 + dist)
        q = q.pow(1.0) # alpha = 1
        q = q / q.sum(dim=1, keepdim=True)
        return q

def get_target_p(q):
    p = q**2 / q.sum(dim=0)
    return p / p.sum(dim=1, keepdim=True)

# A. 预训练阶段
print("\n[方案三] 步骤 A: 预训练卷积自编码器 (50 Epochs)...")
model_ae = ConvAE().to(device)
optimizer_ae = optim.Adam(model_ae.parameters(), lr=0.001)
model_ae.train()
for epoch in range(50):
    for imgs, _ in dataloader:
        imgs = imgs.to(device)
        _, recon = model_ae(imgs)
        loss = F.mse_loss(recon, imgs)
        optimizer_ae.zero_grad()
        loss.backward()
        optimizer_ae.step()

# B. 初始化阶段 (K-means初始化中心)
print("步骤 B: 初始化聚类中心...")
model_ae.eval()
all_z = []
with torch.no_grad():
    for imgs, _ in dataloader:
        z, _ = model_ae(imgs.to(device))
        all_z.append(z.cpu().numpy())
all_z = np.concatenate(all_z)
km_init = KMeans(n_clusters=10, n_init=20).fit(all_z)

# C. 深度联合优化阶段 (DEC)
print("步骤 C: 深度联合优化 (KL Divergence)...")
cluster_layer = ClusteringLayer().to(device)
cluster_layer.centers.data.copy_(torch.tensor(km_init.cluster_centers_))
optimizer_dec = optim.SGD(list(model_ae.encoder.parameters()) + list(cluster_layer.parameters()), lr=0.01, momentum=0.9)

for epoch in range(30):
    for imgs, _ in dataloader:
        imgs = imgs.to(device)
        z = model_ae.encoder(imgs)
        q = cluster_layer(z)
        p = get_target_p(q).detach()

        loss = F.kl_div(q.log(), p, reduction='batchmean')
        optimizer_dec.zero_grad()
        loss.backward()
        optimizer_dec.step()

# D. 最终评估
model_ae.eval()
final_preds = []
with torch.no_grad():
    for imgs, _ in dataloader:
        z = model_ae.encoder(imgs.to(device))
        q = cluster_layer(z)
        final_preds.append(q.argmax(dim=1).cpu().numpy())
y_pred_dec = np.concatenate(final_preds)
acc3, nmi3 = cluster_eval(Y_true, y_pred_dec)
print(f"结果: ACC={acc3:.4f}, NMI={nmi3:.4f}")


[方案三] 步骤 A: 预训练卷积自编码器 (50 Epochs)...
步骤 B: 初始化聚类中心...
步骤 C: 深度联合优化 (KL Divergence)...
结果: ACC=0.2486, NMI=0.1212


# 对比

In [7]:
print("\n" + "="*35)
print(f"方案一 K-means:     ACC={acc1:.4f}")
print(f"方案二 PCA+K-means: ACC={acc2:.4f}")
print(f"方案三 卷积 DEC:    ACC={acc3:.4f}")
print("="*35)


方案一 K-means:     ACC=0.2036
方案二 PCA+K-means: ACC=0.2050
方案三 卷积 DEC:    ACC=0.2486
